# 새 데이터셋 추가하기 — moabb에 없는 로컬 레코딩

직접 딴 레코딩 데이터는 `name=` 대신 `path=`로 불러오는데, `DatasetLoader(path=...)`는 그 폴더 **안에 `dataset_info.py` 메타데이터 파일이 이미 있어야만** 동작합니다 — 없으면 즉시 `FileNotFoundError`를 던집니다. `n2o.signal.dataset.write_metadata_template()`로 빈 템플릿을 직접 생성하고, 채워 넣은 뒤, 레코딩 폴더 안으로 옮겨 넣는 게 여러분이 할 일입니다.


In [1]:
import tempfile
from pathlib import Path

from n2o.signal.dataset import DatasetLoader, write_metadata_template

# 실제로는 진짜 레코딩이 있는 폴더 경로를 쓰면 됩니다 -- 여기서는 노트북을 몇 번을 다시 실행해도
# 항상 깨끗한 상태로 시작하도록 임시 폴더를 하나 만듭니다.
recording_folder = Path(tempfile.mkdtemp()) / "my_lab_emg_recording"
recording_folder.mkdir(parents=True)
print("레코딩 폴더:", recording_folder)

레코딩 폴더: /tmp/tmpq6jm8me6/my_lab_emg_recording


## 1. 메타데이터 없이는 안 됩니다

레코딩 파일들만 있고 `dataset_info.py`가 없는 폴더로 `DatasetLoader`를 만들면, 그 자리에서 바로 거부됩니다.


In [2]:
try:
    DatasetLoader(path=recording_folder)
except FileNotFoundError as e:
    print("의도한 대로 거부됨:", e)

의도한 대로 거부됨: no metadata file at /tmp/tmpq6jm8me6/my_lab_emg_recording/dataset_info.py; call n2o.signal.dataset.write_metadata_template(path) to create one, fill it in, then construct DatasetLoader again


## 2. 메타데이터 템플릿 생성 (레코딩 폴더가 아닌 다른 곳에서)

`write_metadata_template()`은 아무 폴더에나 빈 `DatasetInfo` 템플릿을 씁니다 — 꼭 최종 레코딩 폴더일 필요는 없습니다. 여기서는 일부러 별도의 스크래치 폴더에 만들어서, "어딘가에서 만들고 채운 뒤 나중에 레코딩 폴더로 옮긴다"는 실제 작업 흐름을 그대로 보여줍니다.


In [3]:
scratch_folder = Path(tempfile.mkdtemp())
template_path = write_metadata_template(scratch_folder)
print("템플릿 생성 위치:", template_path)
print()
print(template_path.read_text())

템플릿 생성 위치: /tmp/tmplwfiw7iw/dataset_info.py

"""Metadata for this dataset folder -- generated by
`n2o.signal.dataset.write_metadata_template()`.

Fill in the fields below (delete the TODO comments as you go), place this file inside
the recording folder itself (next to the recording's own files) if it isn't already,
then load it with `DatasetLoader(path=...).info()`. See `n2o.signal.dataset.DatasetInfo`
for what each field means.
"""

from n2o.signal.dataset import DatasetInfo

DATASET_INFO = DatasetInfo(
    source=None,  # TODO: paper/dataset this recording came from (authors, year, DOI)
    cue_onset_sec=None,  # TODO: seconds into a trial where the cue/event appears, if any
    data_range_sec=None,  # TODO: (start, end) seconds relative to the cue this covers
    num_channels=None,  # TODO: number of signal channels
    metadata={
        # TODO: anything else worth recording, e.g. "sampling_rate_hz": ..., "montage": ...
    },
)



## 3. 채워 넣고 레코딩 폴더로 옮기기

실제로는 에디터로 `dataset_info.py`를 열어 TODO를 지우고 값을 채운 뒤, 그 파일을 레코딩 폴더 안으로 옮깁니다. 여기서는 노트북을 그대로 실행할 수 있도록 편집은 코드로, 이동은 `shutil.move()`로 흉내 냅니다.


In [4]:
import shutil

filled = template_path.read_text()
filled = (
    filled.replace(
        "source=None,",
        'source="Park Lab forearm EMG, 4-channel, 2026-08-26",',
    )
    .replace("cue_onset_sec=None,", "cue_onset_sec=1.0,")
    .replace("data_range_sec=None,", "data_range_sec=(0.0, 2.0),")
    .replace("num_channels=None,", "num_channels=4,")
    .replace(
        'metadata={\n        # TODO: anything else worth recording, e.g. "sampling_rate_hz": ..., "montage": ...\n    },',
        'metadata={"sampling_rate_hz": 2000.0, "subject": "S01"},',
    )
)
template_path.write_text(filled)

moved_path = shutil.move(str(template_path), str(recording_folder / "dataset_info.py"))
print("레코딩 폴더로 옮김:", moved_path)

레코딩 폴더로 옮김: /tmp/tmpq6jm8me6/my_lab_emg_recording/dataset_info.py


## 4. 이제 `DatasetLoader`가 동작합니다

메타데이터 파일이 레코딩 폴더 안에 있으니, 같은 `path=`로 만들어도 더 이상 거부되지 않습니다.


In [5]:
info = DatasetLoader(path=recording_folder).info()
info

DatasetInfo(source='Park Lab forearm EMG, 4-channel, 2026-08-26', cue_onset_sec=1.0, data_range_sec=(0.0, 2.0), num_channels=4, metadata={'sampling_rate_hz': 2000.0, 'subject': 'S01'})

## 5. 이미 있는 메타데이터 위에 또 생성하려고 하면?

이미 채워 넣은 파일을 실수로 덮어쓰지 않도록, `write_metadata_template()`은 대상 폴더에 파일이 이미 있으면 거부합니다.


In [6]:
try:
    write_metadata_template(recording_folder)
except FileExistsError as e:
    print("의도한 대로 거부됨:", e)

의도한 대로 거부됨: metadata file already exists at /tmp/tmpq6jm8me6/my_lab_emg_recording/dataset_info.py; delete it first, or construct DatasetLoader(path=...) to load it as-is


이 폴더는 이제 `name="..."`로 등록하지 않고도 `path=`만으로 metadata를 갖춘 데이터셋처럼 다룰 수 있습니다. `.read()`는 `path=` 모드에서도 아직 스텁(`NotImplementedError`)이라 실제 신호 배열을 돌려주진 않습니다 — 이 노트북은 metadata 쪽만 다룹니다.
